In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import os
import subprocess
from math import ceil
from pathlib import Path

import anndata as ad
import colorcet as cc
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import squidpy as sq
from spatialleiden import distance2connectivity, spatialleiden

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_neighbors,
    preprocess_scaling,
)
from utils import (
    DATA_DIR,
    RESULT_DIR,
    SHELL,
    activate_env_cmd,
    mm,
    rc_params,
    remove_tick_and_label,
)

In [ ]:
# Settings
seed = 42

In [ ]:
h5ad = DATA_DIR / "3D_lymphnode" / "GSE251926_metastatic_lymph_node_3d.h5ad"

results_lymphnode = RESULT_DIR / "3D_lymphnode"
processed_h5ad = results_lymphnode / "OpenST_lymphnode.h5ad"

results_lymphnode.mkdir(exist_ok=True, parents=True)

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
mpl.rcParams.update(rc_params)

# Preprocess

In [ ]:
adata = ad.read_h5ad(h5ad)
adata.X = adata.layers["raw"].astype(int)

del adata.layers

In [ ]:
sq.gr.spatial_neighbors(
    adata, coord_type="generic", n_neighs=10, spatial_key="spatial_3d_aligned"
)

In [ ]:
preprocess_genes(adata, batch_key="n_section")
preprocess_scaling(adata, seed=42)

In [ ]:
preprocess_neighbors(adata, "X_pca")

In [ ]:
adata.obsp["spatial_connectivities"] = distance2connectivity(
    adata.obsp["spatial_distances"]
)

In [ ]:
adata.write_h5ad(processed_h5ad)

# Clustering

In [ ]:
spatialleiden(adata, directed=(False, True), layer_ratio=1.8, seed=seed, n_iterations=3)

In [ ]:
columns = ["spatialleiden", "n_section"]

labels = adata.obs[columns]
labels.loc[:, ["x", "y", "z"]] = adata.obsm["spatial_3d_aligned"]
labels.to_parquet(results_lymphnode / "no_batch_integration.parquet")

# Plotting

In [ ]:
scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
labels = pd.read_parquet(results_lymphnode / "no_batch_integration.parquet").astype(
    {"spatialleiden": "category"}
)

In [ ]:
cluster = labels["spatialleiden"].cat.categories

palette = dict(zip(cluster, sns.color_palette(cc.glasbey, n_colors=len(cluster))))

g = sns.FacetGrid(labels, col="n_section", col_wrap=3, aspect=1.2, legend_out=True)
_ = g.map_dataframe(
    sns.scatterplot,
    x="x",
    y="y",
    hue="spatialleiden",
    palette=palette,
    **scatter_kwargs,
)
_ = g.set(aspect=1)
for ax in g.axes.flat:
    remove_tick_and_label(ax)

# g.add_legend(palette)

_ = g.tight_layout()
g.savefig(results_lymphnode / "no_batch_integration.png")

## GIF

In [ ]:
conda_cmd = activate_env_cmd("gif")
script_path = Path("gif") / "gif.py"

In [ ]:
batch_key = "n_section"
fps = 2
height = 7

for file in results_lymphnode.glob("*.parquet"):
    out_file = results_lymphnode / f"{file.stem}.gif"
    cmd = (
        f"{script_path} {file} {out_file} "
        f"--batch {batch_key} "
        f"--fps {fps} "
        f"--height {height} "
    )

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Evaluation


## In and out-of-slice neighbors

In [ ]:
adata = ad.read_h5ad(processed_h5ad)

In [ ]:
from scipy.sparse import csr_matrix

neighbor_stats = []
for g, idx in adata.obs.groupby("n_section").indices.items():
    distances = csr_matrix(adata.obsp["spatial_distances"][idx, :])
    in_slice = distances[:, idx]
    out_of_slice = distances[:, list(set(range(distances.shape[1])) - set(idx))]

    n_in = in_slice.getnnz(axis=1)
    n_out = out_of_slice.getnnz(axis=1)
    mean_in = in_slice.sum(axis=1).A1 / n_in
    mean_out = out_of_slice.sum(axis=1).A1 / n_out

    df = pd.DataFrame(
        {
            "slice": g,
            "mean_dist_in": mean_in,
            "mean_dist_out": mean_out,
            "n_neigh_in": n_in,
            "n_neigh_out": n_out,
        }
    )
    neighbor_stats.append(df)

neighbor_stats = pd.concat(neighbor_stats)

In [ ]:
mean_dist = neighbor_stats.melt(
    "slice", ["mean_dist_in", "mean_dist_out"], "in_slice", "mean_dist"
).assign(
    neighbors=lambda df: df["in_slice"].map(
        {"mean_dist_in": "within slice", "mean_dist_out": "across"}
    )
)
frac_out_neighbors = (
    neighbor_stats.assign(has_out_neighbors=lambda df: df["n_neigh_out"] > 0)
    .groupby(["slice", "has_out_neighbors"], observed=False)
    .size()
    .to_frame("n")
    .reset_index()
    .pivot(columns="has_out_neighbors", index="slice", values="n")
    .assign(fraction=lambda df: df[True] / (df[True] + df[False]))
)

fig, axs = plt.subplots(1, 2, figsize=(180 * mm, 60 * mm), width_ratios=[0.05, 0.95])


_ = sns.boxplot(
    data=frac_out_neighbors,
    y="fraction",
    ax=axs[0],
    flierprops={"marker": ".", "markerfacecolor": "black", "markersize": 3},
)
_ = axs[0].set(
    ylabel="Fraction of cells with neighbors\nin a different slice",
    ylim=(0, 1),
    xticks=[],
)
for x in ["top", "right", "bottom"]:
    axs[0].spines[x].set_visible(False)

_ = sns.boxplot(
    data=mean_dist,
    x="slice",
    y="mean_dist",
    hue="neighbors",
    ax=axs[1],
    fliersize=0,
    gap=0.1,
    flierprops={"marker": ".", "markerfacecolor": "black"},
)
_ = axs[1].set(ylim=(0, 100), ylabel="mean distance [a.u.]")
_ = axs[1].legend_.set(loc="upper left", frame_on=False)

fig.tight_layout()

In [ ]:
n_neigh = neighbor_stats.melt("slice", ["n_neigh_in", "n_neigh_out"], "in_slice", "n")

n = adata.obs["n_section"].nunique()
wrap = int(ceil(n / 2))

aspect = 0.4
height = 180 * mm / wrap / aspect

g = sns.FacetGrid(
    n_neigh.loc[lambda df: df["in_slice"] == "n_neigh_out"],
    col_wrap=10,
    col="slice",
    height=height,
    aspect=aspect,
    sharey=True,
    sharex=False,
)
_ = g.map_dataframe(sns.histplot, y="n", discrete=True)
_ = g.set_titles(col_template="{col_name}")
_ = g.set_axis_labels(
    x_var=r"$\rm n_{cells}$", y_var=r"$\rm n_{neighbors}$ across slices"
)
_ = g.set(yticks=[0, 5, 10])

## Domain compositions

In [ ]:
def stacked_barplot(ax, data, x, y, hue, palette, legend=False):
    import numpy as np

    bottoms = np.zeros(data[x].unique().size)
    x_max = data[x].max() + 0.5
    x_min = data[x].min() - 0.5

    for c, df in data.groupby(hue, observed=False):
        ax.bar(df[x], df[y], 1, label=c, bottom=bottoms, color=palette[c])
        bottoms += df[y].to_numpy()

    ax.set(ylim=(0, 1), xlim=(x_min, x_max), xlabel=x, ylabel=y)
    if legend:
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

In [ ]:
def fraction_of_labels(df, batch, label):
    df = df.groupby([batch, label], observed=False).size().to_frame("n").reset_index()
    df["fraction"] = df["n"] / df.groupby(batch)["n"].transform("sum")
    return df

In [ ]:
adata.obs = adata.obs.join(labels["spatialleiden"])

In [ ]:
batch = "n_section"
anno = "annotation"

frac = fraction_of_labels(adata.obs[[batch, anno]], batch, anno)

fig, axs = plt.subplots(figsize=(4, 2.2))
stacked_barplot(
    axs,
    frac,
    batch,
    "fraction",
    hue=anno,
    palette=dict(zip(adata.obs[anno].cat.categories, cc.glasbey_light)),
    legend=True,
)

In [ ]:
batch = "n_section"
anno = "spatialleiden"

frac = fraction_of_labels(adata.obs[[batch, anno]], batch, anno)

fig, axs = plt.subplots(figsize=(4, 2.2))
stacked_barplot(
    axs,
    frac,
    batch,
    "fraction",
    hue=anno,
    palette=dict(zip(adata.obs[anno].cat.categories, cc.glasbey)),
)

In [ ]:
batch = "spatialleiden"
anno = "annotation"

frac = fraction_of_labels(
    adata.obs[[batch, anno]].assign(**{batch: lambda df: df[batch].cat.as_ordered()}),
    batch,
    anno,
)

fig, axs = plt.subplots(figsize=(4, 2.2))
stacked_barplot(
    axs,
    frac,
    batch,
    "fraction",
    hue=anno,
    palette=dict(zip(adata.obs[anno].cat.categories, cc.glasbey_light)),
    legend=True,
)

In [ ]:
_ = sns.clustermap(
    frac.pivot(columns="annotation", index="spatialleiden", values="fraction")
)